# Chapter 08: Exact Search, ANN, and Index Choice

Exact top-k versus a tiny seeded sign-LSH.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


## Exact cosine reference

In [ ]:
from semantic_search import DenseIndex
records = [
    {"passage_id": "a", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
    {"passage_id": "b", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
    {"passage_id": "c", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
vecs = [[1,0],[0.9,0.1],[0.2,0.8]]
idx = DenseIndex(records, vecs, "m")
q = {"values":[1,0], "model_fingerprint":"m"}
exact = idx.rank(q, 2)
print([(h["passage_id"], round(h["score"],4)) for h in exact])
assert [h["passage_id"] for h in exact] == ["a","b"]

## Tiny seeded sign-LSH

In [ ]:
from semantic_search import SignLSH
lsh = SignLSH(idx, bits=2, tables=1, radius=1, seed=17)
print("candidates:", lsh.candidates(q))
print("lsh:", [h["passage_id"] for h in lsh.rank(q,2)])
assert lsh.rank(q,2) == lsh.rank(q,2)

## Required candidate excluded by approximation

In [ ]:
# Force an empty candidate pool to show no hidden fallback.
lsh2 = SignLSH(idx, bits=2, tables=1, radius=0, seed=17)
lsh2.buckets = [{}]
print("pool result:", lsh2.rank(q,2))
assert lsh2.rank(q,2) == []
assert idx.rank(q,2) != []

## Exact-neighbour versus relevance coverage

In [ ]:
# Exact top-k identity agreement is not evidence coverage.
exact_ids = {h["passage_id"] for h in idx.rank(q,3)}
approx_ids = {h["passage_id"] for h in lsh.rank(q,3)}
print("exact:", exact_ids, "approx:", approx_ids)
print("overlap:", len(exact_ids & approx_ids) / len(exact_ids))
assert len(exact_ids & approx_ids) <= len(exact_ids)

## Try it yourself
Change bits/tables/seed or vectors.

In [ ]:
for bits in [1,2,4]:
    L = SignLSH(idx, bits=bits, tables=1, radius=1, seed=17)
    print("bits", bits, [h["passage_id"] for h in L.rank(q,2)])

## Observed versus not established
Observed: pool exclusion and identity overlap in this toy index. Not established: production ANN behaviour, relevance or helpfulness.